# Automatisk upptäckt av ärendetyper i kundklagomål

**Individuellt projekt – Statistisk programmering och ML (unsupervised learning)**

### Problemformulering
Företag får tusentals kundklagomål i fritext. För att kunna bemanna rätt, utbilda personal och sortera ärenden behöver man veta *vilka typer* av ärenden som kommer in. Men det finns sällan tid att läsa och märka upp texterna manuellt, och de färdiga kategorier som finns (t.ex. vilken produkt klagomålet gäller) säger inte vad kunden faktiskt har för problem.

**Affärsfråga:** Kan vi med unsupervised learning hitta naturliga ärendetyper i kunders klagomålstexter, utan manuell märkning, och identifiera vilka ärendetyper som oftare leder till att kunden får hjälp?

### Varför maskininlärning?
Med över 100 000 texter går det inte att läsa och kategorisera allt för hand. Det finns heller inget facit över vilka ärendetyper som finns. Därför används klustring: modellen grupperar texter som liknar varandra, och grupperna tolkas sedan i efterhand.

### Notebookens upplägg
1. Datakälla och inläsning
2. Urval och rensning
3. EDA av texterna *(kommande)*

## 1. Datakälla och inläsning

**Datakälla:** CFPB Consumer Complaint Database, den amerikanska konsumentskyddsmyndighetens öppna databas över klagomål på finansiella produkter och tjänster.

**Viktigt om datan:** den vanliga nedladdningen från CFPB innehåller inte längre klagomålstexterna. Texterna finns i stället i myndighetens [Narratives Archive](https://www.consumerfinance.gov/foia-requests/foia-electronic-reading-room/cfpb-consumer-complaint-database-narratives-archive/), som är uppdelat i zip-filer per tidsperiod.

Den senaste arkivfilen (augusti 2026) testades först, men innehöll bara **1 klagomål med text av 667 140**. Därför används en äldre period: **november 2022 – augusti 2023**.

Filen laddades ner 2026-11-02 och ligger i `data/`, som inte laddas upp till GitHub (för stor).

In [ ]:
import pandas as pd

FIL = "../data/CCDB_Export_4_November_2022_through_August_2023.zip"

### 1.1 Hur många klagomål har text?

Konsumenten väljer själv om klagomålstexten får publiceras, så alla rader har inte text. Innan vi läser in allt kontrollerar vi hur stor andel som har text och om texterna är jämnt fördelade över perioden.

Filen läses i bitar om 200 000 rader, så att datorn inte får slut på minne.

In [ ]:
totalt = 0
med_text = 0
datum = []

for chunk in pd.read_csv(FIL, usecols=["Date received", "Consumer complaint narrative"],
                         chunksize=200_000, low_memory=False):
    totalt += len(chunk)
    har_text = chunk["Consumer complaint narrative"].notna()
    med_text += har_text.sum()
    datum.append(pd.to_datetime(chunk.loc[har_text, "Date received"]))

datum = pd.concat(datum)
print("Totalt antal rader:", totalt)
print("Med text:", med_text, f"({med_text / totalt:.1%})")
print("Texter finns från", datum.min().date(), "till", datum.max().date())
print("\nAntal texter per månad:")
print(datum.dt.to_period("M").value_counts().sort_index())

**Resultat:** 388 495 av 987 495 klagomål (39,3 %) har text. Texterna är jämnt fördelade över alla tio månader (ca 31 000–46 000 per månad), utan luckor. En arkivfil räcker därför för projektet.

**Begränsning:** eftersom konsumenterna själva väljer att publicera sin text kan de skilja sig från dem som inte gör det (urvalsproblem).

### 1.2 Läs in klagomålen med text

Bara de kolumner som behövs läses in:

| Kolumn | Används till |
|---|---|
| `Consumer complaint narrative` | Själva texten som ska klustras |
| `Product`, `Sub-product`, `Issue`, `Sub-issue` | Konsumentens egen kategorisering, används för att jämföra med klustren |
| `Company response to consumer` | Utfall: fick kunden hjälp? |
| `Date received`, `Company`, `State`, `Submitted via` | Beskriva och tolka klustren |
| `Timely response?` | Om företaget svarade i tid |
| `Complaint ID` | Unikt id |

Resultatet sparas som en parquet-fil, ett komprimerat format som går mycket snabbare att läsa in nästa gång.

In [ ]:
cols = [
    "Date received", "Product", "Sub-product", "Issue", "Sub-issue",
    "Consumer complaint narrative", "Company", "State",
    "Submitted via", "Company response to consumer",
    "Timely response?", "Complaint ID",
]

chunks = []
for chunk in pd.read_csv(FIL, usecols=cols, chunksize=200_000, low_memory=False):
    chunks.append(chunk.dropna(subset=["Consumer complaint narrative"]))

df = pd.concat(chunks, ignore_index=True)
df["Date received"] = pd.to_datetime(df["Date received"])

df.to_parquet("../data/complaints_narratives.parquet")

print("Antal klagomål:", len(df))
print("\nAndel per produkt:")
print(df["Product"].value_counts(normalize=True).round(3))
print("\nTimely response?:\n", df["Timely response?"].value_counts(normalize=True).round(3))
print("\nCompany response:\n", df["Company response to consumer"].value_counts())

**Tre observationer som styr rensningen:**

1. **Kreditupplysningar dominerar.** De två kategorierna som börjar med "Credit reporting" står tillsammans för ca 72 % av klagomålen. De skulle dominera klustringen och dölja övriga ärendetyper.
2. **`Timely response?` har nästan ingen variation.** 99,4 % fick svar i tid, så kolumnen kan inte skilja ärendetyper åt. I stället används `Company response to consumer` som utfall.
3. **Produktnamnen är inte enhetliga.** CFPB har bytt namn på kategorier över tid, t.ex. finns både "Credit card" och "Credit card or prepaid card".

## 2. Urval och rensning

### 2.1 Ta bort kreditupplysningar och slå ihop produktnamn

**Kreditupplysningar tas bort** eftersom de handlar nästan uteslutande om tvister med tre kreditbyråer och har en annan karaktär än vanliga kundserviceärenden. Kvar blir klagomål om bankkonton, kort, inkasso, lån och betalningar.

**Produktnamn som betyder samma sak slås ihop** så att samma produkt inte räknas som två kategorier.

Vi räknar också hur många texter som är exakta dubbletter (samma text inskickad flera gånger).

In [ ]:
df = pd.read_parquet("../data/complaints_narratives.parquet")

# Ta bort kreditupplysningar (båda namnvarianterna)
df = df[~df["Product"].str.startswith("Credit reporting")]

# Slå ihop produktnamn som bytt namn över tid
produkt_map = {
    "Credit card": "Credit card or prepaid card",
    "Prepaid card": "Credit card or prepaid card",
    "Payday loan, title loan, personal loan, or advance loan":
        "Payday loan, title loan, or personal loan",
}
df["Product"] = df["Product"].replace(produkt_map)

# Räkna dubbletter
dubbletter = df.duplicated(subset="Consumer complaint narrative").sum()

print("Antal klagomål kvar:", len(df))
print("Exakta dubbletter av text:", dubbletter, f"({dubbletter / len(df):.1%})")
print("\nAntal per produkt:")
print(df["Product"].value_counts())
print("\nCompany response:")
print(df["Company response to consumer"].value_counts())

df.to_parquet("../data/complaints_filtered.parquet")

**Resultat:** 107 916 klagomål kvar, fördelade på åtta produkter med rimlig balans (störst är inkasso med ca 28 500, minst studielån med ca 2 900). 5,0 % av texterna är exakta dubbletter, troligen mallbrev.

### 2.2 Slutlig rensning och utfallsvariabel

| Åtgärd | Motivering |
|---|---|
| Ta bort exakta dubbletter (behåll första) | Identiska texter skulle bilda konstgjorda, extremt täta kluster och ge en missvisande bild av vad kunderna klagar på. |
| Ta bort "Debt or credit management" | Endast 22 klagomål, för få för att säga något om. |
| Ta bort "Untimely response" | 235 klagomål saknar besked om hur ärendet avslutades. |

**Utfallsvariabel `fick_hjalp`:**
- `1` = företaget gav kunden ekonomisk kompensation (*monetary relief*) eller annan hjälp (*non-monetary relief*)
- `0` = ärendet avslutades bara med en förklaring

Variabeln används **inte** i klustringen. Den används i efterhand för att undersöka om vissa ärendetyper oftare leder till att kunden får rätt.

In [ ]:
df = pd.read_parquet("../data/complaints_filtered.parquet")

df = df.drop_duplicates(subset="Consumer complaint narrative", keep="first")
df = df[df["Product"] != "Debt or credit management"]
df = df[df["Company response to consumer"] != "Untimely response"]

df["fick_hjalp"] = df["Company response to consumer"].isin(
    ["Closed with monetary relief", "Closed with non-monetary relief"]
).astype(int)

print("Antal klagomål:", len(df))
print("Andel som fick hjälp:", df["fick_hjalp"].mean().round(3))
print("\nAndel som fick hjälp per produkt:")
print(df.groupby("Product")["fick_hjalp"].mean().sort_values(ascending=False).round(3))

df.to_parquet("../data/complaints_clean.parquet")

**Resultat:** 102 247 klagomål återstår, och 17,5 % av kunderna fick någon form av hjälp.

**Första insikt:** chansen att få hjälp skiljer sig kraftigt mellan produkter. Klagomål om kort leder till hjälp i 32 % av fallen, bolån bara i 5 %. Samma fråga ställs senare om klustren: skiljer sig ärendetyperna mer än produkterna gör?

> **Statistisk anmärkning:** med över 100 000 rader blir nästan varje skillnad statistiskt signifikant. När chi²-test görs redovisas därför också effektstorleken (Cramér's V), som visar hur *starkt* sambandet är.

### Sammanfattning av urvalet

| Steg | Antal klagomål |
|---|---|
| Arkivfil nov 2022 – aug 2023 | 987 495 |
| Med klagomålstext | 388 495 |
| Utan kreditupplysningar | 107 916 |
| Efter borttagning av dubbletter, "Debt or credit management" och "Untimely response" | 102 247 |

Den rensade datan sparas i `data/complaints_clean.parquet` och är utgångspunkten för EDA och modellering.